In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import os
import re
import torch.utils.data as data

In [ ]:
class TextDataset(data.Dataset):
    def __init__(self, path, seq_length, step=3):
        self.seq_length = seq_length
        self.step = step # <-- Новый параметр: шаг сдвига
        
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read()
        text = text.replace('\ufeff', '')
        
        # Чистка (опционально, но полезно)
        # text = re.sub(r'[^А-яA-z0-9.,?;: \n]', '', text) 

        self.chars = sorted(list(set(text)))
        self.char_to_int = {c: i for i, c in enumerate(self.chars)}
        self.int_to_char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars) 

        self.encoded_text = torch.tensor([self.char_to_int[c] for c in text], dtype=torch.long)

    def __len__(self):
        # Математика: (Длина - Окно) / Шаг
        return (len(self.encoded_text) - self.seq_length) // self.step

    def __getitem__(self, idx):
        # Индекс умножаем на шаг
        start_idx = idx * self.step
        
        chunk = self.encoded_text[start_idx : start_idx + self.seq_length + 1]
        
        input_seq = chunk[:-1] 
        target_seq = chunk[1:]
        
        return input_seq, target_seq

In [ ]:
d_train = TextDataset('pushkin.txt', 128, step=3)
n = len(d_train)
train_size = int(n*0.8)
val_size = int(n*0.1)
teset_size = int(n*0.1)
train_indices = range(0, train_size)
val_indices = range(train_size, train_size + val_size)
test_indices = range(train_size + val_size, n)
train_ds = data.Subset(d_train, train_indices)
val_ds = data.Subset(d_train, val_indices)
test_ds = data.Subset(d_train, test_indices)
train_loader = data.DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True)
val_loader = data.DataLoader(val_ds, batch_size=256, shuffle=False, drop_last=True)
test_loader = data.DataLoader(test_ds, batch_size=256, shuffle=False, drop_last=True)

In [85]:
import torch
import torch.nn as nn

class CharGRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256, n_layers=2):
        super(CharGRU, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        self.gru = nn.GRU(
            input_size=emb_dim, 
            hidden_size=hidden_dim, 
            num_layers=n_layers, 
            batch_first=True,
            dropout=0.2 
        )
        
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

    def forward(self, x, hidden):

        embedded = self.embedding(x)

        out, hidden = self.gru(embedded, hidden)
        
        prediction = self.fc(out)
        
        return prediction, hidden

    def init_hidden(self, batch_size, device):
        # Эта функция нужна, чтобы создать нулевую память для начала обучения
        return torch.zeros(self.n_layers, batch_size, self.hidden_dim).to(device)

In [92]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
print(device)
# Гиперпараметры
VOCAB_SIZE = len(d_train.chars) 
EMB_DIM = 128
HIDDEN_DIM = 512
N_LAYERS = 3
BATCH_SIZE = 256
LR = 0.001       
EPOCHS = 8    

mps


In [93]:
def generate(model, start_str='Я', predict_len=100, temperature=0.8):
    model.eval() # Переключаем в режим оценки (выключаем Dropout)
    
    # 1. Превращаем начальную строку в индексы
    chars = [c for c in start_str]
    input_seq = torch.tensor([d_train.char_to_int.get(c, 0) for c in chars]).unsqueeze(0).to(device)
    
    # 2. Инициализируем память
    hidden = model.init_hidden(1, device)
    
    predicted_text = start_str
    
    # 3. "Прогреваем" память на начальной строке
    # Нам не важен выход, важно, чтобы hidden напитался контекстом
    for i in range(len(start_str) - 1):
        _, hidden = model(input_seq[:, i:i+1], hidden)
    
    # Последний символ start_str — это наш первый вход для генерации
    input_char = input_seq[:, -1:]
    
    with torch.no_grad(): # Градиенты не нужны
        for _ in range(predict_len):
            output, hidden = model(input_char, hidden)
            
            # output: (1, 1, vocab_size) -> (vocab_size)
            output_logits = output[0, 0] 
            
            # Применяем температуру (чем выше, тем больше бреда/креатива)
            probs = torch.softmax(output_logits / temperature, dim=0)
            
            # Сэмплируем (выбираем) следующую букву с учетом вероятностей
            char_idx = torch.multinomial(probs, 1).item()
            
            predicted_char = d_train.int_to_char[char_idx]
            predicted_text += predicted_char
            
            # Этот символ становится входом для следующего шага
            input_char = torch.tensor([[char_idx]]).to(device)
            
    return predicted_text

In [94]:
# Инициализация модели
model = CharGRU(VOCAB_SIZE, EMB_DIM, HIDDEN_DIM, N_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


for epoch in range(EPOCHS):
    model.train() 
    train_loss = 0
    loop = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    for i, (inputs, targets) in enumerate(loop):

        inputs, targets = inputs.to(device), targets.to(device)
        

        hidden = model.init_hidden(BATCH_SIZE, device)
        

        optimizer.zero_grad()
        
        output, hidden = model(inputs, hidden)
        
        loss = criterion(output.view(-1, VOCAB_SIZE), targets.view(-1))
        
        loss.backward()
        

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)

        optimizer.step()
        
        train_loss += loss.item()

    # В конце эпохи выводим статистику и пример текста
    avg_loss = train_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}')
    print("Пример:", generate(model, start_str="Зима", predict_len=100))
    print("-" * 50)

Epoch 1/8: 100%|██████████| 205/205 [02:31<00:00,  1.35it/s]


Epoch 1/8 | Loss: 1.9724
Пример: Зимали, стара словстры?
        Тапьян, развитным, в отмонный,
        Позы мастолонне моженья,
        
--------------------------------------------------


Epoch 2/8: 100%|██████████| 205/205 [02:37<00:00,  1.31it/s]


Epoch 2/8 | Loss: 1.2926
Пример: Зимательных бемним,
        Там колено стекрыв оставить
        И стра насладились с ней,
        Чтоб н
--------------------------------------------------


Epoch 3/8: 100%|██████████| 205/205 [02:36<00:00,  1.31it/s]


...

Epoch 6/8 | Loss: 0.3033
Пример: Зиманим волномал,
        И одно изнывая смиреньем
        Простите мне: я так люблю
        Татьяну мил
--------------------------------------------------


Epoch 7/8: 100%|██████████| 205/205 [02:40<00:00,  1.27it/s]


Epoch 7/8 | Loss: 0.2361
Пример: Зимательном далее меня».
        
        XXXIV
        
        Тут непременно вы найдёте
        Два с
--------------------------------------------------


Epoch 8/8: 100%|██████████| 205/205 [02:42<00:00,  1.26it/s]


Epoch 8/8 | Loss: 0.1998
Пример: Зиман,
        Что модных колец не достали.
        О свадьбе Ленского давно
        У них уж было решен
--------------------------------------------------
